# Diedros

**Módulo 06 — Geometria Espacial** · Notebook 05 de 18

[![Abrir no Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/leandrofreiredealmeida/curso_matematica_python/blob/main/06_geometria_espacial/0605a_diedros.ipynb)

**Pré-requisitos**
- Módulo 06 — Geometria Espacial: [`0603a_perpendicularidade.ipynb`](0603a_perpendicularidade.ipynb) (plano perpendicular a uma reta, planos perpendiculares) e [`0604a_projecoes_distancias_e_angulos.ipynb`](0604a_projecoes_distancias_e_angulos.ipynb) (reta de maior declive, planos bissetores, projeção ortogonal)
- Módulo 04 — Geometria Plana: [`0403a_angulos.ipynb`](../04_geometria_plana/0403a_angulos.ipynb) (congruência de ângulos, ângulos complementares e suplementares, ângulos opostos pelo vértice e bissetriz)
- Módulo 05 — Trigonometria: [`0509a_triangulos_retangulos.ipynb`](../05_trigonometria/0509a_triangulos_retangulos.ipynb) (razões trigonométricas no triângulo retângulo)

**Objetivos de aprendizagem** — ao final deste notebook você será capaz de:
- identificar um diedro, seus elementos e suas seções;
- medir um diedro pela sua seção reta e classificá-lo;
- determinar o semiplano bissetor e o ângulo entre dois planos;
- usar o ângulo entre planos para resolver problemas reais, como a área de um telhado.

**Tempo estimado:** 60 a 80 minutos

### Antes de começar

Rode a célula abaixo antes de qualquer outra. Ela carrega o `utils_3d.py`, o arquivo com as funções de desenho deste módulo.

In [1]:
# Célula de preparação: deixa o notebook pronto para rodar no seu computador ou no Colab.
import sys

if "google.colab" in sys.modules:
    # No Colab, o notebook chega sozinho; baixamos o arquivo de desenhos do repositório.
    import urllib.request
    url = "https://raw.githubusercontent.com/leandrofreiredealmeida/curso_matematica_python/main/06_geometria_espacial/utils_3d.py"
    urllib.request.urlretrieve(url, "utils_3d.py")

import utils_3d as u3d
from ipywidgets import interact
import ipywidgets as widgets

## Qual é o ângulo de abertura da tela?

Você abre um notebook (o computador!) e quer saber o ângulo entre a tela e o teclado. Pega um transferidor, encosta na dobradiça e lê o ângulo. Simples?

Nem tanto. Dependendo de **como** você posiciona o transferidor, a leitura muda! Encostado de um jeito, ele mostra 100°; girado um pouco, mostra 110°, 120°... Qual é a resposta certa?

🕹️ No widget abaixo, a tela está aberta num certo ângulo. Gire o transferidor (o plano em que você faz a medição) e veja a leitura mudar.

In [2]:
import math

# As coordenadas (x, y, z) abaixo servem só para posicionar o desenho.


def medir_a_tela(abertura_da_tela: int, giro_do_transferidor: int) -> None:
    """Desenha o notebook aberto e o transferidor, e mostra a leitura."""
    d = u3d.vertices_do_diedro(abertura_da_tela, comprimento=6, largura=3)
    diedro = (d["aresta_inicio"], d["aresta_fim"], d["ponto_face1"], d["ponto_face2"])

    # Seno e cosseno (módulo 05) servem só para girar o desenho do transferidor em
    # volta da vertical: com giro 0, ele fica perpendicular à dobradiça.
    g = math.radians(giro_do_transferidor)
    transferidor = ((0, 0, 0), (math.cos(g), math.sin(g), 0))
    vertice, lado1, lado2 = u3d.secao_do_diedro(*diedro, transferidor)
    leitura = u3d.medir_angulo(vertice, lado1, lado2)

    fig = u3d.criar_figura(titulo="Medindo a abertura da tela")
    fig = u3d.desenhar_diedro(
        fig, *diedro, rotulos=("teclado", "tela"), rotulo_aresta="dobradiça",
        cores=(u3d.CORES["solido"], u3d.CORES["plano"]), largura=3,
    )
    fig = u3d.desenhar_secao_do_diedro(
        fig, *diedro, transferidor, rotulo=f"{leitura:.1f}°", comprimento=2.5,
        opacidade_plano=0.25,
    )
    u3d.mostrar(fig)

    print(f"Leitura do transferidor: {leitura:.1f}°")
    if giro_do_transferidor == 0:
        print(
            "Transferidor perpendicular à dobradiça: esta é a medida verdadeira "
            "da abertura."
        )
    else:
        print("Transferidor inclinado: a leitura não é a abertura verdadeira.")


interact(
    medir_a_tela,
    abertura_da_tela=widgets.IntSlider(
        min=60, max=150, step=10, value=100, description="abertura da tela (°)",
        continuous_update=False,
    ),
    giro_do_transferidor=widgets.IntSlider(
        min=-60, max=60, step=10, value=30, description="giro do transferidor (°)",
        continuous_update=False,
    ),
);

interactive(children=(IntSlider(value=100, continuous_update=False, description='abertura da tela (°)', max=15…

A abertura entre a tela e o teclado é um **diedro**, e para medi-lo corretamente o transferidor precisa estar **perpendicular à dobradiça**. Neste notebook você vai entender por que essa é a medição certa e vai usar diedros para calcular, por exemplo, quantas telhas são necessárias para cobrir um telhado.

## 1. Semiplanos e diedros

Na Geometria Plana, um ponto divide uma reta em duas **semirretas**, e duas semirretas de mesma origem formam um **ângulo**. No espaço, a ideia "sobe" uma dimensão:

> 📌 **Definição.**
> - Uma reta $r$ de um plano divide esse plano em duas partes; cada parte, junto com a reta $r$, é um **semiplano** de **origem** $r$.
> - Da mesma forma, um plano divide o espaço em dois **semiespaços**.
> - Um **diedro** é a figura formada por dois semiplanos de mesma origem que não estão contidos num mesmo plano.

**Elementos e notação.** No diedro $\mathrm{di}(\alpha\, r\, \beta)$:
- a reta $r$ (origem comum) é a **aresta**;
- os semiplanos $\alpha$ e $\beta$ são as **faces**.

| No plano (ângulo) | No espaço (diedro) |
|---|---|
| semirreta | semiplano |
| vértice | aresta |
| lados | faces |

**Diedros no dia a dia:** um livro aberto (a lombada é a aresta), um notebook aberto, uma porta e a parede, as duas águas de um telhado (a cumeeira é a aresta), a quina entre duas paredes.

**Diedros adjacentes e opostos pela aresta.** Assim como os ângulos:
- dois diedros são **adjacentes** quando têm a mesma aresta e uma face em comum, sem sobreposição;
- quando dois planos se cortam, eles formam **quatro** diedros; os que não são adjacentes são **opostos pela aresta**.

🕹️ Mude a abertura do diedro e escolha o que mostrar.

In [3]:
OPCOES_DE_DIEDROS = [
    "Um diedro",
    "Dois diedros adjacentes",
    "Os quatro diedros de dois planos",
]


def ponto_em_volta_da_aresta(angulo: float, largura: float = 4) -> tuple:
    """Devolve um ponto a `largura` da aresta, numa face que forma `angulo` com α."""
    # Seno e cosseno (módulo 05) servem só para girar o ponto em volta da aresta.
    t = math.radians(angulo)
    return (0, largura * math.cos(t), largura * math.sin(t))


def mostrar_diedros(abertura: int, mostrar: str) -> None:
    """Desenha um diedro, dois diedros adjacentes ou os quatro diedros."""
    d = u3d.vertices_do_diedro(abertura)
    a, b = d["aresta_inicio"], d["aresta_fim"]
    p1, p2 = d["ponto_face1"], d["ponto_face2"]

    fig = u3d.criar_figura(titulo=mostrar)
    if mostrar == "Um diedro":
        fig = u3d.desenhar_diedro(
            fig, a, b, p1, p2, rotulos=("α", "β"), rotulo_aresta="r"
        )
        u3d.mostrar(fig)
        print(f"di(α, r, β) = {u3d.medir_angulo_do_diedro(a, b, p1, p2):.0f}°")

    elif mostrar == "Dois diedros adjacentes":
        # A terceira face γ gira mais `abertura` graus a partir de β, sem passar de
        # 180° no total (senão os dois diedros se sobreporiam).
        p3 = ponto_em_volta_da_aresta(min(2 * abertura, 180))
        fig = u3d.desenhar_diedro(
            fig, a, b, p1, p2, rotulos=("α", "β"), rotulo_aresta="r"
        )
        fig = u3d.desenhar_semiplano(
            fig, a, b, p3, rotulo="γ", cor=u3d.CORES["plano_terciario"]
        )
        u3d.mostrar(fig)
        print(f"di(α, r, β) = {u3d.medir_angulo_do_diedro(a, b, p1, p2):.0f}°")
        print(f"di(β, r, γ) = {u3d.medir_angulo_do_diedro(a, b, p2, p3):.0f}°")
        print("Os dois diedros têm a aresta r e a face β em comum: são adjacentes.")

    else:
        # As faces opostas: os pontos simétricos de p1 e p2 em relação à aresta.
        q1 = (0, -p1[1], -p1[2])
        q2 = (0, -p2[1], -p2[2])
        faces = [(p1, "α"), (p2, "β"), (q1, "α'"), (q2, "β'")]
        # Cores alternadas em volta da aresta: cada plano tem uma cor só.
        cores = [u3d.CORES["plano"], u3d.CORES["plano_secundario"]] * 2
        for (ponto, rotulo), cor in zip(faces, cores, strict=True):
            fig = u3d.desenhar_semiplano(fig, a, b, ponto, rotulo=rotulo, cor=cor)
        fig = u3d.desenhar_reta(
            fig, (0, 0, 0), (1, 0, 0), rotulo="r", cor=u3d.CORES["destaque"],
            comprimento=6, espessura=6,
        )
        u3d.mostrar(fig)
        # Cada diedro é formado por duas faces vizinhas, em volta da aresta.
        for indice in range(4):
            (ponto1, nome1), (ponto2, nome2) = faces[indice], faces[(indice + 1) % 4]
            medida = u3d.medir_angulo_do_diedro(a, b, ponto1, ponto2)
            print(f"di({nome1}, r, {nome2}) = {medida:.0f}°")


interact(
    mostrar_diedros,
    abertura=widgets.IntSlider(
        min=20, max=160, step=10, value=70, description="abertura (°)",
        continuous_update=False,
    ),
    mostrar=widgets.Dropdown(
        options=OPCOES_DE_DIEDROS, value="Um diedro", description="mostrar"
    ),
);

interactive(children=(IntSlider(value=70, continuous_update=False, description='abertura (°)', max=160, min=20…

**Pare e pense**

1. Quantos diedros dois planos secantes formam? Quais deles são congruentes?
2. No cubo, quantos diedros têm como aresta uma aresta do cubo e como faces duas faces do cubo?

<details>
<summary>Ver respostas</summary>

1. Quatro. Os opostos pela aresta são congruentes (como os ângulos opostos pelo vértice).
2. Doze, um para cada aresta do cubo.
</details>

## 2. Seções de um diedro

> 📌 **Definição.**
> - Uma **seção** de um diedro é o ângulo obtido cortando o diedro por um plano que corta a aresta num único ponto.
> - Uma **seção reta** (ou **seção normal**) é uma seção feita por um plano **perpendicular à aresta**. O ângulo obtido é o **ângulo retilíneo** do diedro.

**Teorema**
> Todas as seções retas de um mesmo diedro são congruentes.

<details>
<summary>Ver demonstração</summary>

Sejam $\pi_1$ e $\pi_2$ dois planos perpendiculares à aresta $r$, cortando-a em $O_1$ e $O_2$. Pela propriedade 4 da seção 3 do notebook 03, $\pi_1 \parallel \pi_2$.

As seções retas são os ângulos $x_1 O_1 y_1$ e $x_2 O_2 y_2$. As semirretas $O_1 x_1$ e $O_2 x_2$ estão na mesma face $\alpha$, ambas perpendiculares a $r$; logo são paralelas e apontam para o mesmo lado. O mesmo vale para $O_1 y_1$ e $O_2 y_2$ na face $\beta$.

Dois ângulos cujos lados são paralelos e de mesmo sentido são congruentes (o segundo é uma "translação" do primeiro ao longo de $r$). ∎
</details>

> ⚠️ **Erro comum.** Medir um diedro com uma seção **oblíqua**. Ela pode dar um ângulo **maior ou menor** que a abertura verdadeira, como você viu no gancho. Só a seção reta mede o diedro.

🎯 **Aplicação.** É assim que se mede a inclinação de um telhado ou de um painel solar: o instrumento de medida (um transferidor, um inclinômetro) precisa ficar num plano perpendicular à linha de encontro dos dois planos (a cumeeira, a borda de apoio).

🕹️ Mova o plano de corte ao longo da aresta e incline-o. Compare as leituras com a medida do diedro.

In [4]:
def leitura_da_secao(abertura: int, posicao: float, inclinacao_do_corte: int) -> float:
    """Devolve a medida da seção do diedro pelo plano de corte escolhido."""
    d = u3d.vertices_do_diedro(abertura, comprimento=9)
    diedro = (d["aresta_inicio"], d["aresta_fim"], d["ponto_face1"], d["ponto_face2"])
    # Seno e cosseno (módulo 05) servem só para inclinar o desenho do plano de corte:
    # com inclinação 0, ele é perpendicular à aresta.
    g = math.radians(inclinacao_do_corte)
    corte = ((posicao, 0, 0), (math.cos(g), math.sin(g), 0))
    return u3d.medir_angulo(*u3d.secao_do_diedro(*diedro, corte))


def explorar_secoes(abertura: int, posicao: float, inclinacao_do_corte: int) -> None:
    """Desenha o diedro e a seção pelo plano de corte, com a tabela de leituras."""
    d = u3d.vertices_do_diedro(abertura, comprimento=9)
    diedro = (d["aresta_inicio"], d["aresta_fim"], d["ponto_face1"], d["ponto_face2"])
    # Seno e cosseno (módulo 05) servem só para inclinar o desenho (veja acima).
    g = math.radians(inclinacao_do_corte)
    corte = ((posicao, 0, 0), (math.cos(g), math.sin(g), 0))
    leitura = leitura_da_secao(abertura, posicao, inclinacao_do_corte)
    medida = u3d.medir_angulo_do_diedro(*diedro)

    fig = u3d.criar_figura(titulo="Seções de um diedro")
    fig = u3d.desenhar_diedro(
        fig, *diedro, rotulos=("α", "β"), rotulo_aresta="r", comprimento=9
    )
    fig = u3d.desenhar_secao_do_diedro(
        fig, *diedro, corte, rotulo=f"{leitura:.1f}°", comprimento=2.5,
        opacidade_plano=0.25,
    )
    u3d.mostrar(fig)

    print(f"Leitura da seção: {leitura:.1f}°")
    print(f"Medida do diedro: {medida:.1f}°")
    if inclinacao_do_corte == 0:
        print("Seção reta: leitura = medida do diedro, em qualquer posição.")

    print()
    print(f"Diedro de {abertura}°:")
    print(f"{'inclinação do corte':^22}{'leitura da seção':^20}")
    for inclinacao in range(-60, 61, 20):
        valor = leitura_da_secao(abertura, posicao, inclinacao)
        print(f"{f'{inclinacao}°':^22}{f'{valor:.1f}°':^20}")


interact(
    explorar_secoes,
    abertura=widgets.IntSlider(
        min=30, max=150, step=10, value=80, description="abertura (°)",
        continuous_update=False,
    ),
    posicao=widgets.FloatSlider(
        min=-2, max=2, step=0.5, value=0, description="posição do corte",
        continuous_update=False,
    ),
    inclinacao_do_corte=widgets.IntSlider(
        min=-60, max=60, step=10, value=0, description="inclinação do corte (°)",
        continuous_update=False,
    ),
);

interactive(children=(IntSlider(value=80, continuous_update=False, description='abertura (°)', max=150, min=30…

**Pare e pense**

1. Na tabela, a leitura de uma seção oblíqua foi sempre maior que a medida do diedro? Sempre menor?
2. No cubo, desenhe mentalmente uma seção reta do diedro de aresta $AE$ formado pelas faces $ABFE$ e $ADHE$. Que ângulo você obtém?

<details>
<summary>Ver respostas</summary>

1. Depende da abertura do diedro: neste widget, para diedros agudos as seções oblíquas diminuem a leitura, e para diedros obtusos aumentam (para o diedro reto, com esse tipo de giro, a leitura nem muda). Como a leitura depende do corte, a medida precisa ser definida por um corte fixo: a seção reta.
2. Um plano perpendicular a $AE$ (horizontal) corta as faces em retas paralelas a $AB$ e a $AD$: a seção é um ângulo reto. O diedro mede 90°.
</details>

## 3. Medida, congruência e bissetor

> 📌 **Definição.**
> - A **medida** de um diedro é a medida do seu ângulo retilíneo (de qualquer seção reta).
> - Dois diedros são **congruentes** quando têm a mesma medida.

**Classificação**

| Medida | Diedro |
|---|---|
| menor que 90° | **agudo** |
| igual a 90° | **reto** (as faces estão em planos perpendiculares, notebook 03) |
| maior que 90° | **obtuso** |

Dois diedros são **complementares** quando suas medidas somam 90°, e **suplementares** quando somam 180°. Dois diedros adjacentes cujas faces não comuns formam um plano são suplementares (pense numa porta aberta encostada numa parede reta: o diedro de um lado da porta e o do outro lado somam 180°).

> 📌 **Definição.** O **semiplano bissetor** de um diedro é o semiplano com origem na aresta que divide o diedro em dois diedros adjacentes congruentes.

**Propriedades do bissetor**
- A seção reta do bissetor é a **bissetriz** do ângulo retilíneo.
- Todo ponto do semiplano bissetor está à **mesma distância** das duas faces. É por isso que, no notebook 04, os pontos equidistantes de dois planos secantes formavam os **planos bissetores**.

🕹️ Veja abaixo o bissetor e um ponto $P$ sobre ele. Mexa na abertura e na posição de $P$: as distâncias às faces continuam iguais.

In [5]:
def explorar_bissetor(abertura: int, posicao_de_P: float) -> None:
    """Desenha o diedro, o semiplano bissetor e as distâncias de P às faces."""
    d = u3d.vertices_do_diedro(abertura)
    a, b = d["aresta_inicio"], d["aresta_fim"]
    p1, p2 = d["ponto_face1"], d["ponto_face2"]
    q = u3d.ponto_do_semiplano_bissetor(a, b, p1, p2)
    plano_alfa = u3d.plano_por_tres_pontos(a, b, p1)
    plano_beta = u3d.plano_por_tres_pontos(a, b, p2)

    # O ponto q está a distância 1 do ponto a, sobre o bissetor. Andando a partir do
    # ponto O = (0, 0, 0) da aresta na mesma direção, chegamos ao ponto P do bissetor
    # que está a `posicao_de_P` da aresta, na seção reta que passa por O.
    O = (0, 0, 0)
    direcao_do_bissetor = [qi - ai for qi, ai in zip(q, a, strict=True)]
    P = tuple(posicao_de_P * c for c in direcao_do_bissetor)
    pe_alfa = u3d.pe_da_perpendicular_ao_plano(P, plano_alfa)
    pe_beta = u3d.pe_da_perpendicular_ao_plano(P, plano_beta)

    fig = u3d.criar_figura(titulo="O semiplano bissetor")
    fig = u3d.desenhar_diedro(
        fig, a, b, p1, p2, rotulos=("α", "β"), rotulo_aresta="r", opacidade=0.25
    )
    fig = u3d.desenhar_semiplano(
        fig, a, b, q, rotulo="bissetor", cor=u3d.CORES["plano_terciario"],
        opacidade=0.3,
    )
    # A seção reta pelo ponto O: os lados nas faces e a bissetriz no bissetor.
    fig = u3d.desenhar_secao_do_diedro(
        fig, a, b, p1, p2, ((0, 0, 0), (1, 0, 0)), comprimento=4,
        mostrar_plano=False, mostrar_arco=False,
    )
    fig = u3d.desenhar_segmento(fig, O, tuple(4 * c for c in direcao_do_bissetor))
    metade = f"{abertura / 2:.0f}°"
    fig = u3d.desenhar_angulo(fig, O, p1, P, rotulo=metade, raio=1)
    fig = u3d.desenhar_angulo(fig, O, P, p2, rotulo=metade, raio=1)
    # As distâncias de P às faces: segmentos perpendiculares, com o ângulo reto no pé.
    for pe in (pe_alfa, pe_beta):
        fig = u3d.desenhar_segmento(
            fig, P, pe, cor=u3d.CORES["destaque"], espessura=4, tracejado=True
        )
        fig = u3d.desenhar_angulo_reto(fig, pe, O, P, tamanho=0.25)
    fig = u3d.desenhar_ponto(fig, P, rotulo="P", cor=u3d.CORES["destaque"])
    fig = u3d.definir_camera(fig, olho=(1.9, 0.6, 0.9))
    u3d.mostrar(fig)

    print(f"Distância de P até α: {u3d.medir_distancia_ponto_plano(P, plano_alfa):.3f}")
    print(f"Distância de P até β: {u3d.medir_distancia_ponto_plano(P, plano_beta):.3f}")
    medida1 = u3d.medir_angulo_do_diedro(a, b, p1, q)
    medida2 = u3d.medir_angulo_do_diedro(a, b, q, p2)
    print(f"di(α, r, bissetor) = {medida1:.1f}°")
    print(f"di(bissetor, r, β) = {medida2:.1f}°")


interact(
    explorar_bissetor,
    abertura=widgets.IntSlider(
        min=30, max=150, step=10, value=100, description="abertura (°)",
        continuous_update=False,
    ),
    posicao_de_P=widgets.FloatSlider(
        min=0.5, max=3.5, step=0.5, value=2, description="distância de P à aresta",
        continuous_update=False,
    ),
);

interactive(children=(IntSlider(value=100, continuous_update=False, description='abertura (°)', max=150, min=3…

### Diedros no cubo

🕹️ Escolha um diedro do cubo, tente descobrir sua medida e depois confira. A figura mostra a seção reta usada na medição.

In [6]:
vertices = u3d.vertices_do_cubo()

# Cada diedro: as duas letras da aresta, uma letra da face 1 e uma letra da face 2.
DIEDROS_DO_CUBO = {
    "Aresta AB: faces ABCD e ABFE": "ABDE",
    "Aresta AB: faces ABCD e ABGH": "ABDG",
    "Aresta AC: faces ABC e ACGE": "ACBG",
    "Aresta BD: faces BDA e BDE": "BDAE",
    "Aresta BD: faces BDC e BDE": "BDCE",
    "Aresta BF: faces ABFE e BCGF": "BFAC",
}


def classificar_diedro(medida: float) -> str:
    """Classifica um diedro pela sua medida, em graus."""
    if abs(medida - 90) < 0.05:
        return "reto"
    return "agudo" if medida < 90 else "obtuso"


def explorar_diedro_do_cubo(diedro: str, mostrar_resposta: bool) -> None:
    """Desenha o diedro escolhido no cubo e, se pedido, a sua seção reta."""
    a, b, p1, p2 = (vertices[letra] for letra in DIEDROS_DO_CUBO[diedro])
    meio = tuple((ai + bi) / 2 for ai, bi in zip(a, b, strict=True))
    tamanho_da_aresta = u3d.medir_distancia(a, b)

    fig = u3d.criar_figura(titulo=diedro)
    fig = u3d.desenhar_cubo(fig)
    # Pedaços pequenos das faces, para caberem dentro do cubo.
    fig = u3d.desenhar_diedro(
        fig, a, b, p1, p2, comprimento=0.5 * tamanho_da_aresta, largura=0.7,
        opacidade=0.5, mostrar_aresta=False,
    )
    fig = u3d.desenhar_segmento(fig, a, b, cor=u3d.CORES["destaque"], espessura=6)
    if mostrar_resposta:
        medida = u3d.medir_angulo_do_diedro(a, b, p1, p2)
        # A seção reta: o plano perpendicular à aresta pelo seu ponto médio.
        corte = u3d.plano_perpendicular_a_reta(a, b, meio)
        fig = u3d.desenhar_secao_do_diedro(
            fig, a, b, p1, p2, corte, rotulo=f"{medida:.1f}°", comprimento=0.6,
            opacidade_plano=0.3,
        )
    u3d.mostrar(fig)

    if mostrar_resposta:
        print(f"Medida do diedro: {medida:.1f}° (diedro {classificar_diedro(medida)})")
    else:
        print("Pense na seção reta e marque a caixa para conferir.")


interact(
    explorar_diedro_do_cubo,
    diedro=widgets.Dropdown(options=list(DIEDROS_DO_CUBO), description="diedro"),
    mostrar_resposta=widgets.Checkbox(value=False, description="Mostrar resposta"),
);

interactive(children=(Dropdown(description='diedro', options=('Aresta AB: faces ABCD e ABFE', 'Aresta AB: face…

**Pare e pense**

1. Um diedro mede 35°. Quanto mede o seu complemento? E o seu suplemento?
2. No widget, os diedros de aresta $BD$ com faces $BDA$/$BDE$ e $BDC$/$BDE$ somam quanto? Por quê?

<details>
<summary>Ver respostas</summary>

1. Complemento: 55°. Suplemento: 145°.
2. 180°. São diedros adjacentes (face comum $BDE$), e as faces não comuns ($BDA$ e $BDC$) formam o plano da base. Logo são suplementares.
</details>

## 4. Ângulo entre dois planos

Dois planos secantes formam quatro diedros: dois de medida $\theta$ e dois de medida $180° - \theta$.

> 📌 **Definição.** O **ângulo entre dois planos** é:
> - $0°$, se forem paralelos ou coincidentes;
> - a medida do **menor** dos diedros que eles formam, se forem secantes (entre $0°$ e $90°$).

Planos perpendiculares formam ângulo de $90°$.

**Duas formas de medir o ângulo entre planos**
1. **Pela reta de maior declive** (notebook 04): o ângulo entre um plano $\alpha$ e um plano $\beta$ é o ângulo entre $\beta$ e uma reta de maior declive de $\alpha$. De fato, a reta de maior declive é perpendicular à interseção, e sua projeção em $\beta$ também é: as duas formam uma seção reta do diedro.
2. **Por retas perpendiculares aos planos:** o ângulo entre dois planos é igual ao ângulo entre duas retas perpendiculares a eles. (No plano da seção reta, as perpendiculares às faces formam um ângulo igual ao retilíneo ou ao seu suplemento, pois cada uma está "girada" de 90° em relação a uma face.)

### A área da projeção

**Teorema**
> Se uma figura plana de área $S$ está num plano que forma ângulo $\theta$ com um plano $\beta$, a área $S'$ da sua projeção ortogonal sobre $\beta$ é
> $$S' = S \cdot \cos\theta$$

**Ideia da justificativa.** Para um retângulo com um lado paralelo à interseção dos planos, esse lado se projeta em verdadeira grandeza, e o outro lado (na direção da reta de maior declive) fica multiplicado por $\cos\theta$ (notebook 04). A área fica multiplicada por $\cos\theta$. Uma figura qualquer pode ser aproximada por retângulos assim, cada vez menores.

🎯 **Aplicação: quantas telhas comprar?** A planta baixa de uma casa é a projeção do telhado sobre o chão. Conhecendo a área da planta $S'$ e a inclinação $\theta$ do telhado, a área real do telhado é
$$S = \frac{S'}{\cos\theta}$$
É com essa área que o pedreiro calcula a quantidade de telhas.

> 💡 **Dica.** Painéis solares também usam essa ideia: eles costumam ser instalados com uma inclinação próxima da latitude do local, para que fiquem mais "de frente" para o sol ao longo do ano. A inclinação do painel é o ângulo entre o plano do painel e o plano do chão.

🕹️ Monte o telhado de duas águas abaixo e calcule a área real dele.

In [ ]:
ALTURA_DAS_PAREDES = 3  # em metros


def desenhar_telhado(comprimento: int, largura: int, inclinacao: int) -> None:
    """Desenha a casa com telhado de duas águas e calcula a área do telhado."""
    c, h = comprimento, ALTURA_DAS_PAREDES
    # Tangente (módulo 05): a cumeeira fica (largura / 2) · tan θ acima do beiral.
    altura_da_cumeeira = h + (largura / 2) * math.tan(math.radians(inclinacao))
    meio = largura / 2
    chao = ((c / 2, meio, 0), (0, 0, 1))

    paredes = [
        [(0, 0, 0), (c, 0, 0), (c, 0, h), (0, 0, h)],
        [(c, 0, 0), (c, largura, 0), (c, largura, h), (c, 0, h)],
        [(c, largura, 0), (0, largura, 0), (0, largura, h), (c, largura, h)],
        [(0, largura, 0), (0, 0, 0), (0, 0, h), (0, largura, h)],
    ]
    # Os oitões: os triângulos das paredes de frente e de fundo, sob o telhado.
    oitoes = [
        [(0, 0, h), (0, largura, h), (0, meio, altura_da_cumeeira)],
        [(c, 0, h), (c, largura, h), (c, meio, altura_da_cumeeira)],
    ]
    agua1 = [(0, 0, h), (c, 0, h), (c, meio, altura_da_cumeeira),
             (0, meio, altura_da_cumeeira)]
    agua2 = [(0, largura, h), (c, largura, h), (c, meio, altura_da_cumeeira),
             (0, meio, altura_da_cumeeira)]

    fig = u3d.criar_figura(titulo="Telhado de duas águas", altura=550)
    fig = u3d.desenhar_plano(
        fig, *chao, rotulo="chão", opacidade=0.15, tamanho=max(c, largura) + 4
    )
    for parede in paredes + oitoes:
        fig = u3d.desenhar_poligono(fig, parede, opacidade=0.2)
    fig = u3d.desenhar_poligono(fig, agua1, cor=u3d.CORES["plano"], opacidade=0.6)
    fig = u3d.desenhar_poligono(
        fig, agua2, cor=u3d.CORES["plano_secundario"], opacidade=0.6
    )
    # A planta baixa: a projeção das duas águas no chão.
    for agua in (agua1, agua2):
        fig = u3d.desenhar_projecao_ortogonal(
            fig, agua, chao, fechar=True, preencher=True, mostrar_projetantes=False
        )
    # A seção reta de uma água, no meio do comprimento: o ângulo θ fica no beiral,
    # entre a água e o plano horizontal que passa pelo beiral.
    beiral = (c / 2, 0, h)
    fig = u3d.desenhar_segmento(
        fig, beiral, (c / 2, meio, h), cor=u3d.CORES["apoio"], espessura=3,
        tracejado=True,
    )
    fig = u3d.desenhar_segmento(
        fig, (c / 2, meio, h), (c / 2, meio, altura_da_cumeeira),
        cor=u3d.CORES["apoio"], espessura=3, tracejado=True,
    )
    fig = u3d.desenhar_angulo(
        fig, beiral, (c / 2, 1, h), (c / 2, meio, altura_da_cumeeira),
        rotulo=f"{inclinacao}°", raio=1.2,
    )
    fig = u3d.definir_camera(fig, olho=(1.3, -1.6, 0.8))
    u3d.mostrar(fig)

    area_planta = comprimento * largura
    area_telhado = area_planta / math.cos(math.radians(inclinacao))

    # Conferência pelo caminho "direto": cada água é um retângulo
    largura_da_agua = (largura / 2) / math.cos(math.radians(inclinacao))
    area_direta = 2 * comprimento * largura_da_agua

    plano_da_agua = u3d.plano_por_tres_pontos(*agua1[:3])
    angulo_conferido = u3d.medir_angulo_entre_planos(plano_da_agua, chao)
    print(f"Área da planta (projeção): {area_planta:.2f} m²")
    print(f"Área do telhado (planta / cos θ): {area_telhado:.2f} m²")
    print(f"Área do telhado (somando as duas águas): {area_direta:.2f} m²")
    print(f"Ângulo entre a água e o chão (conferido): {angulo_conferido:.1f}°")


interact(
    desenhar_telhado,
    comprimento=widgets.IntSlider(
        min=6, max=14, step=1, value=10, description="comprimento (m)",
        continuous_update=False,
    ),
    largura=widgets.IntSlider(
        min=4, max=10, step=1, value=8, description="largura (m)",
        continuous_update=False,
    ),
    inclinacao=widgets.IntSlider(
        min=10, max=45, step=5, value=30, description="inclinação (°)",
        continuous_update=False,
    ),
);

interactive(children=(IntSlider(value=10, continuous_update=False, description='comprimento (m)', max=14, min=…

**Pare e pense**

1. Um telhado com 30° de inclinação cobre uma planta de 80 m². A área do telhado é maior ou menor que 80 m²? Calcule.
2. O que acontece com a área do telhado quando a inclinação se aproxima de 90°? Faz sentido?

<details>
<summary>Ver respostas</summary>

1. Maior: $\frac{80}{\cos 30°} \approx 92{,}4$ m².
2. Ela cresce sem limite, porque $\cos\theta$ se aproxima de 0. Faz sentido: um "telhado" quase vertical precisaria ser altíssimo para cobrir a mesma planta.
</details>

## 5. Seções igualmente inclinadas e congruência de diedros

A seção reta usa duas semirretas que formam **90°** com a aresta. E se as duas semirretas formarem com a aresta um **mesmo** ângulo $\varphi$, diferente de 90°?

> 📌 **Definição.** Uma **seção igualmente inclinada** de um diedro é um ângulo $x\hat{O}y$, com $O$ na aresta, $Ox$ numa face e $Oy$ na outra, tal que $Ox$ e $Oy$ formam com a aresta, **do mesmo lado**, ângulos congruentes. Esse ângulo comum $\varphi$ é a **inclinação** da seção.

A seção reta é a seção igualmente inclinada com $\varphi = 90°$.

**Teorema**
> Dois diedros são congruentes se, e somente se, têm seções igualmente inclinadas congruentes, com a mesma inclinação.

Em outras palavras: para comparar dois diedros, você não precisa da seção reta. Basta usar, nos dois, seções igualmente inclinadas com a mesma inclinação $\varphi$. Se essas seções forem iguais, os diedros são iguais (e vice-versa).

📎 Esse resultado será a ferramenta principal do notebook 06, para comparar **triedros**.

🕹️ **Explorando com números.** O widget abaixo mostra uma seção igualmente inclinada. A tabela abaixo da figura calcula a seção para várias aberturas do diedro, mantendo a inclinação fixa. Repare: para uma inclinação fixa, diedros diferentes sempre dão seções diferentes, e diedros maiores dão seções maiores. É isso que garante o teorema.

In [8]:
VERTICE_DA_SECAO = (-1.5, 0, 0)  # um ponto da aresta do diedro padrão


def angulo_da_secao_inclinada(abertura: int, inclinacao: int) -> float:
    """Devolve a medida da seção igualmente inclinada, em graus."""
    d = u3d.vertices_do_diedro(abertura)
    diedro = (d["aresta_inicio"], d["aresta_fim"], d["ponto_face1"], d["ponto_face2"])
    secao = u3d.secao_igualmente_inclinada(*diedro, VERTICE_DA_SECAO, inclinacao)
    return u3d.medir_angulo(*secao)


def explorar_secao_inclinada(abertura: int, inclinacao: int) -> None:
    """Desenha o diedro e a seção igualmente inclinada, com a tabela de seções."""
    d = u3d.vertices_do_diedro(abertura)
    diedro = (d["aresta_inicio"], d["aresta_fim"], d["ponto_face1"], d["ponto_face2"])
    vertice, lado1, lado2 = u3d.secao_igualmente_inclinada(
        *diedro, VERTICE_DA_SECAO, inclinacao
    )
    angulo = u3d.medir_angulo(vertice, lado1, lado2)

    fig = u3d.criar_figura(titulo="Uma seção igualmente inclinada")
    fig = u3d.desenhar_diedro(
        fig, *diedro, rotulos=("α", "β"), rotulo_aresta="r", opacidade=0.25
    )
    # lado1 e lado2 estão a distância 1 do vértice: esticamos as semirretas até 3.
    for lado in (lado1, lado2):
        ponta = tuple(v + 3 * (p - v) for v, p in zip(vertice, lado, strict=True))
        fig = u3d.desenhar_segmento(
            fig, vertice, ponta, cor=u3d.CORES["reta"], espessura=5
        )
        # A inclinação φ: o ângulo entre a semirreta e a aresta (no sentido de a→b).
        fig = u3d.desenhar_angulo(
            fig, vertice, d["aresta_fim"], lado, rotulo="φ", cor=u3d.CORES["apoio"],
            raio=0.5,
        )
    fig = u3d.desenhar_angulo(
        fig, vertice, lado1, lado2, rotulo=f"{angulo:.1f}°", raio=1.2
    )
    fig = u3d.desenhar_ponto(fig, vertice, rotulo="O")
    u3d.mostrar(fig)

    print(f"Medida do diedro: {u3d.medir_angulo_do_diedro(*diedro):.1f}°")
    print(f"Inclinação φ: {inclinacao}°")
    print(f"Ângulo da seção: {angulo:.1f}°")

    # Com a inclinação fixa, os valores da tabela crescem junto com a abertura:
    # diedros diferentes dão seções diferentes. Então seções iguais ⟹ diedros iguais.
    print()
    print(f"Inclinação fixa φ = {inclinacao}°:")
    print(f"{'abertura do diedro':^22}{'ângulo da seção':^20}")
    for abertura_da_tabela in range(30, 151, 30):
        valor = angulo_da_secao_inclinada(abertura_da_tabela, inclinacao)
        print(f"{f'{abertura_da_tabela}°':^22}{f'{valor:.1f}°':^20}")


interact(
    explorar_secao_inclinada,
    abertura=widgets.IntSlider(
        min=30, max=150, step=10, value=90, description="abertura (°)",
        continuous_update=False,
    ),
    inclinacao=widgets.IntSlider(
        min=15, max=90, step=15, value=45, description="inclinação φ (°)",
        continuous_update=False,
    ),
);

interactive(children=(IntSlider(value=90, continuous_update=False, description='abertura (°)', max=150, min=30…

**Pare e pense**

1. Com inclinação de 90°, qual é a relação entre o ângulo da seção e a medida do diedro?
2. Dois diedros têm seções igualmente inclinadas de 60°, com inclinação de 45°. O que você pode concluir sobre os diedros?

<details>
<summary>Ver respostas</summary>

1. São iguais: com $\varphi = 90°$, a seção igualmente inclinada é a seção reta.
2. Que eles são congruentes (pelo teorema).
</details>

## Resumo

**Diedro** $\mathrm{di}(\alpha\, r\, \beta)$: dois semiplanos ($\alpha$, $\beta$, as **faces**) de mesma origem $r$ (a **aresta**), não contidos num mesmo plano.

**Seções:** corte por um plano que encontra a aresta num ponto. **Seção reta** = corte por plano perpendicular à aresta = **ângulo retilíneo**. Todas as seções retas de um diedro são congruentes; seções oblíquas podem ser maiores ou menores.

**Medida:** a medida do retilíneo. Agudo ($< 90°$), reto ($= 90°$, faces perpendiculares), obtuso ($> 90°$). Complementares: somam 90°. Suplementares: somam 180°. Opostos pela aresta: congruentes.

**Bissetor:** semiplano que divide o diedro em dois congruentes; seus pontos são equidistantes das faces.

**Ângulo entre dois planos:** 0° (paralelos) ou o menor diedro formado (secantes), entre 0° e 90°. É igual ao ângulo da reta de maior declive e ao ângulo entre retas perpendiculares aos planos.

**Área da projeção:** $S' = S \cos\theta$. Telhado: $S_{\text{telhado}} = \dfrac{S_{\text{planta}}}{\cos\theta}$.

**Seção igualmente inclinada:** lados com a mesma inclinação $\varphi$ em relação à aresta. Diedros congruentes ⟺ seções igualmente inclinadas congruentes de mesma inclinação.

## Exercícios

Responda preenchendo as variáveis indicadas em cada célula (troque cada `...` pela sua resposta) e rode a célula de verificação logo abaixo. Os exercícios estão em três níveis: **básico**, **intermediário** e **desafio**. Nos exercícios com o cubo, use aresta 2.

O gabarito comentado está em [`0605b_diedros_gabarito.ipynb`](0605b_diedros_gabarito.ipynb). Tente sozinho antes de olhar!

Use o cubo abaixo para consulta durante os exercícios. Gire a figura à vontade!

In [9]:
fig = u3d.criar_figura(titulo="Cubo ABCDEFGH (aresta 2)")
fig = u3d.desenhar_cubo(fig)
u3d.mostrar(fig)

### Básico

In [ ]:
# Exercício Básico 1: verdadeiro ou falso? Troque cada ... por True ou False.
# (a) Um diedro é formado por dois semiplanos de mesma origem, não contidos num
#     mesmo plano.
# (b) Todas as seções de um diedro têm a mesma medida.
# (c) Todas as seções retas de um diedro são congruentes.
# (d) Dois diedros opostos pela aresta são congruentes.
# (e) Se um diedro mede 90°, suas faces estão em planos perpendiculares.
respostas_ex1 = {
    "a": ...,
    "b": ...,
    "c": ...,
    "d": ...,
    "e": ...,
}

print(respostas_ex1)

In [ ]:
# Verificação
gabarito_ex1 = {"a": True, "b": False, "c": True, "d": True, "e": True}
secoes_ex1 = {
    "a": "seção 1 - Semiplanos e diedros",
    "b": "seção 2 - Seções de um diedro",
    "c": "seção 2 - Seções de um diedro",
    "d": "seção 1 - Semiplanos e diedros",
    "e": "seção 3 - Medida, congruência e bissetor",
}
for item, certa in gabarito_ex1.items():
    assert respostas_ex1[item] == certa, (
        f"Tente de novo, revise: o item ({item}) não está certo; releia as definições "
        f"e os teoremas ({secoes_ex1[item]})."
    )
print("Certo!")

In [ ]:
# Exercício Básico 2: medida, em graus, de cada diedro do cubo de aresta 2.
# (a) aresta AB, faces ABCD e ABFE
# (b) aresta AB, faces ABCD e ABGH
# (c) aresta AE, faces ABFE e ADHE
# (d) aresta AC, faces ABC (a parte da base do lado de B) e ACGE
diedros_ex2 = {
    "a": ...,
    "b": ...,
    "c": ...,
    "d": ...,
}

print(diedros_ex2)

In [ ]:
# Verificação
gabarito_ex2 = {"a": 90, "b": 45, "c": 90, "d": 90}
for item, certa in gabarito_ex2.items():
    resposta = diedros_ex2[item]
    assert resposta is not ... and abs(resposta - certa) <= 0.1, (
        f"Tente de novo, revise: no item ({item}), imagine a seção reta: um plano "
        "perpendicular à aresta passando pelo seu ponto médio (seção 2 - Seções de "
        "um diedro)."
    )
print("Certo!")

In [ ]:
# Exercício Básico 3: responda em graus.
# (a) o complemento de um diedro de 35°
# (b) o suplemento de um diedro de 35°
# (c) a medida de cada diedro formado pelo semiplano bissetor de um diedro de 128°
respostas_ex3 = {
    "a": ...,
    "b": ...,
    "c": ...,
}

print(respostas_ex3)

In [ ]:
# Verificação
gabarito_ex3 = {"a": 55, "b": 145, "c": 64}
for item, certa in gabarito_ex3.items():
    resposta = respostas_ex3[item]
    assert resposta is not ... and abs(resposta - certa) <= 0.1, (
        f"Tente de novo, revise: no item ({item}), lembre que complementares somam "
        "90°, suplementares somam 180°, e o bissetor divide ao meio (seção 3 - "
        "Medida, congruência e bissetor)."
    )
print("Certo!")

### Intermediário

In [ ]:
# Exercício Intermediário 4: uma casa tem planta retangular de 8 m × 10 m e telhado
# de duas águas com inclinação de 30°. Calcule em Python (não escreva só o
# resultado):
# (a) a área do telhado, em m²;
# (b) o número de telhas, sabendo que são necessárias 16 telhas por m²
#     (arredonde para cima com math.ceil).
largura_da_planta = 8
comprimento_da_planta = 10
inclinacao = 30
telhas_por_m2 = 16

area_telhado_ex4 = ...  # (a)
telhas_ex4 = ...  # (b)

print(area_telhado_ex4, telhas_ex4)

In [ ]:
# Verificação
assert area_telhado_ex4 is not ... and abs(area_telhado_ex4 - 92.376) <= 0.01, (
    "Tente de novo, revise: área do telhado = área da planta / cos(inclinação); "
    "lembre de converter graus em radianos (seção 4 - Ângulo entre dois planos)."
)
assert telhas_ex4 is not ... and telhas_ex4 == 1479, (
    "Tente de novo, revise: número de telhas = área do telhado × 16, arredondado "
    "para cima com math.ceil (seção 4 - Ângulo entre dois planos)."
)
print("Certo!")

In [ ]:
# Exercício Intermediário 5: calcule a medida, em graus, do diedro de aresta BD
# formado pela base (do lado do vértice A) e pelo plano BDE. Escreva o cálculo em
# Python.
# Dica: seja O o ponto médio de BD. O ângulo AÔE é a seção reta (por quê?). No
# triângulo AOE, retângulo em A, use a tangente.
aresta = 2

diedro_ex5 = ...

print(diedro_ex5)

In [ ]:
# Verificação
assert diedro_ex5 is not ... and abs(diedro_ex5 - 54.7356) <= 0.1, (
    "Tente de novo, revise: AE = 2 e AO = metade da diagonal da base = √2; "
    "tan θ = AE / AO (seção 2 - Seções de um diedro)."
)
print("Certo!")

# Conferência com o utils_3d (que usa vetores, assunto do módulo 07).
v = u3d.vertices_do_cubo()
medida = u3d.medir_angulo_do_diedro(v["B"], v["D"], v["A"], v["E"])
print(f"Medida conferida: {medida:.4f}°")

In [ ]:
# Exercício Intermediário 6: a tela de um notebook forma com o teclado um diedro
# de 110°.
# (a) Qual é o ângulo entre os planos da tela e do teclado?
# (b) Se a tela abrir mais 20°, qual será o novo ângulo entre os planos?
angulo_planos_ex6a = ...
angulo_planos_ex6b = ...

print(angulo_planos_ex6a, angulo_planos_ex6b)

In [ ]:
# Verificação
assert angulo_planos_ex6a is not ... and abs(angulo_planos_ex6a - 70) <= 0.1, (
    "Tente de novo, revise: o ângulo entre planos é o menor dos diedros formados, "
    "sempre entre 0° e 90° (seção 4 - Ângulo entre dois planos)."
)
assert angulo_planos_ex6b is not ... and abs(angulo_planos_ex6b - 50) <= 0.1, (
    "Tente de novo, revise: com a tela aberta a 130°, o ângulo entre planos é o "
    "menor dos diedros formados, sempre entre 0° e 90° (seção 4 - Ângulo entre "
    "dois planos)."
)
print("Certo!")

### Desafio

In [ ]:
# Exercício Desafio 7: os planos ABGH e ADGF contêm a diagonal AG do cubo.
# (a) Qual é a medida, em graus, do diedro de aresta AG com faces passando por B e
#     por D?
# (b) Qual é o ângulo, em graus, entre os planos ABGH e ADGF?
# Dica: no notebook 03 você viu que AG ⊥ BDE e que AG fura esse plano num ponto P.
# Então o plano BDE dá uma seção reta desse diedro. Que tipo de triângulo é BDE, e
# onde fica P dentro dele?
diedro_ex7 = ...  # (a)
angulo_planos_ex7 = ...  # (b)

print(diedro_ex7, angulo_planos_ex7)

In [ ]:
# Verificação
assert diedro_ex7 is not ... and abs(diedro_ex7 - 120) <= 0.1, (
    "Tente de novo, revise: BDE é um triângulo equilátero (seus lados são diagonais "
    "de faces) e P é o seu centro; qual é o ângulo BPD? (seção 2 - Seções de um "
    "diedro)."
)
assert angulo_planos_ex7 is not ... and abs(angulo_planos_ex7 - 60) <= 0.1, (
    "Tente de novo, revise: o ângulo entre planos é o menor dos diedros que eles "
    "formam, entre 0° e 90° (seção 4 - Ângulo entre dois planos)."
)
print("Certo!")

# Conferência com o utils_3d (que usa vetores, assunto do módulo 07).
v = u3d.vertices_do_cubo()
A, B, D, E, G = v["A"], v["B"], v["D"], v["E"], v["G"]
plano_abgh = u3d.plano_por_tres_pontos(A, B, G)
plano_adgf = u3d.plano_por_tres_pontos(A, D, G)
P = u3d.intersecao_reta_plano(A, G, u3d.plano_por_tres_pontos(B, D, E))
print(f"Diedro medido: {u3d.medir_angulo_do_diedro(A, G, B, D):.1f}°")
angulo = u3d.medir_angulo_entre_planos(plano_abgh, plano_adgf)
print(f"Ângulo entre os planos: {angulo:.1f}°")

fig = u3d.criar_figura(titulo="A seção reta BPD do diedro de aresta AG")
fig = u3d.desenhar_cubo(fig)
# Dentro do cubo, os dois semiplanos de origem AG aparecem como os triângulos ABG e
# ADG.
fig = u3d.desenhar_poligono(fig, [A, B, G], cor=u3d.CORES["plano"], opacidade=0.35)
fig = u3d.desenhar_poligono(
    fig, [A, D, G], cor=u3d.CORES["plano_secundario"], opacidade=0.35
)
fig = u3d.desenhar_segmento(fig, A, G, cor=u3d.CORES["destaque"], espessura=6)
fig = u3d.desenhar_poligono(
    fig, [B, D, E], cor=u3d.CORES["plano_terciario"], opacidade=0.25
)
fig = u3d.desenhar_segmento(fig, P, B, cor=u3d.CORES["reta"], espessura=5)
fig = u3d.desenhar_segmento(fig, P, D, cor=u3d.CORES["reta"], espessura=5)
fig = u3d.desenhar_angulo(fig, P, B, D, rotulo="120°", raio=0.4)
fig = u3d.desenhar_ponto(fig, P, rotulo="P", cor=u3d.CORES["destaque"])
u3d.mostrar(fig)
print(
    "O plano ACGE também contém AG: os três planos dividem o espaço em volta de AG "
    "em seis diedros de 60°."
)

In [ ]:
# Exercício Desafio 8: para um diedro reto (90°), qual inclinação φ dá uma seção
# igualmente inclinada de exatamente 60°? Complete o laço abaixo: ele testa as
# inclinações de 1° a 90° e guarda aquela cuja seção fica mais perto de 60°.
diedro = u3d.vertices_do_diedro(90)
a, b = diedro["aresta_inicio"], diedro["aresta_fim"]
p1, p2 = diedro["ponto_face1"], diedro["ponto_face2"]
vertice = (0, 0, 0)  # um ponto da aresta

melhor_inclinacao_ex8 = ...
menor_diferenca = 1000  # começa bem grande: qualquer diferença real será menor

for inclinacao in range(1, 91):
    # Use u3d.secao_igualmente_inclinada(a, b, p1, p2, vertice, inclinacao)
    secao = ...  # complete aqui
    # Meça o ângulo da seção com u3d.medir_angulo(*secao)
    angulo = ...  # complete aqui
    # Se abs(angulo - 60) for menor que menor_diferenca, guarde essa diferença e
    # a inclinação em melhor_inclinacao_ex8
    ...  # complete aqui

print(melhor_inclinacao_ex8)

In [ ]:
# Verificação
assert melhor_inclinacao_ex8 is not ... and abs(melhor_inclinacao_ex8 - 45) <= 0.5, (
    "Tente de novo, revise: guarde, para cada inclinação, a diferença entre o ângulo "
    "da seção e 60°, e escolha a menor (seção 5 - Seções igualmente inclinadas e "
    "congruência de diedros)."
)
print("Certo!")

## Conexões

**Isso depende de:**
- [`0603a_perpendicularidade.ipynb`](0603a_perpendicularidade.ipynb) (plano perpendicular a uma reta, planos perpendiculares)
- [`0604a_projecoes_distancias_e_angulos.ipynb`](0604a_projecoes_distancias_e_angulos.ipynb) (reta de maior declive, planos bissetores, projeção ortogonal)
- [`0403a_angulos.ipynb`](../04_geometria_plana/0403a_angulos.ipynb) (congruência, ângulos complementares e suplementares, opostos pelo vértice e bissetriz)
- [`0509a_triangulos_retangulos.ipynb`](../05_trigonometria/0509a_triangulos_retangulos.ipynb) (razões trigonométricas no triângulo retângulo)

**Isso será usado em:**
- 📌 Triedros e ângulos poliédricos (diedros de um triedro, seções igualmente inclinadas)
- 📌 Poliedros convexos (diedros entre faces dos poliedros de Platão)
- 📌 Pirâmide (diedro entre a face lateral e a base)
- 📌 Esfera (a cunha esférica é limitada por um diedro)
- 📌 Geometria Analítica Plana e Espacial (ângulo entre planos pelos vetores normais)